# 2-2절 연습 문제 풀이

이 노트북은 2-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch02/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 2-4

torch.randn() 함수로 만든 텐서의 요솟값은 평균 0, 표준편차 1인 정규분포를 따르며, 각 요소의 값이 -2~2 사이일 확률은 95%이다. 초깃값이 평균 0, 표준편차 10인 정규분포를 따르도록 가중치 초기화를 바꾸면 학습 결과가 어떻게 될까? 초기화 방법을 바꿔 서로 다른 초깃값으로 10번 이상 반복 학습해 결과를 확인해 보자.

힌트: 표준편차가 10인 값으로 초기화한 텐서가 파라미터로 사용하기 적절한지 확인해 볼 필요가 있다.

In [ ]:
# 표준편차가 큰 초깃값이 학습에 어떤 영향을 주는지 10회 반복해 확인한다.
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [0.], [0.], [1.]])          # AND 게이트

def train_perceptron(std, epochs=2000, lr=0.1, seed=0):
    g = torch.Generator().manual_seed(seed)
    w = (torch.randn(2, 1, generator=g) * std).requires_grad_(True)
    b = (torch.randn(1, generator=g) * std).requires_grad_(True)
    for _ in range(epochs):
        pred = torch.sigmoid(X @ w + b)
        loss = ((pred - Y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad; b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
    correct = ((torch.sigmoid(X @ w + b) > 0.5).float() == Y).all().item()
    return loss.item(), correct

for std in (1.0, 10.0):
    losses = [train_perceptron(std, seed=s) for s in range(10)]
    ok = sum(1 for _, c in losses if c)
    print(f'표준편차 {std:4.1f} -> 10회 중 정답 {ok}회, '
          f'평균 손실 {sum(l for l, _ in losses) / 10:.4f}')

표준편차가 10이면 초기 가중합의 절댓값이 커져 시그모이드가 0 또는 1에 **포화**된다. 포화 구간에서는 기울기가 0에 가까워 파라미터가 거의 갱신되지 않고(기울기 소실), 학습이 시작조차 못 하거나 매우 느려진다.

초깃값은 활성화 함수가 민감하게 반응하는 구간(시그모이드는 0 부근)에 머물도록 **작게** 잡아야 한다. 이 원리를 체계화한 것이 8장에서 다루는 배치 정규화와 가중치 초기화 기법이다.

## 연습 2-5

OR 게이트와 NAND 게이트를 시뮬레이션하는 모델을 각각 만들어 보자.

힌트: OR 게이트는 두 개의 입력이 모두 0일 때만 출력이 0이고, NAND 게이트는 두 입력이 모두 1일 때만 출력이 0이다.

In [ ]:
def train_gate(X, Y, epochs=3000, lr=0.5):
    model = nn.Sequential(nn.Linear(X.size(1), 1), nn.Sigmoid())
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
gates = {
    'OR':   torch.tensor([[0.], [1.], [1.], [1.]]),
    'NAND': torch.tensor([[1.], [1.], [1.], [0.]]),
}
for name, Y in gates.items():
    model = train_gate(X, Y)
    pred = (model(X) > 0.5).float()
    print(f'[{name}] 예측 {pred.flatten().tolist()} / 정답 {Y.flatten().tolist()}'
          f' -> {"성공" if torch.equal(pred, Y) else "실패"}')

OR와 NAND는 하나의 직선으로 0과 1을 나눌 수 있는 **선형 분리 가능** 문제라 퍼셉트론 하나로 학습된다. 정답 텐서만 바꾸면 같은 구조로 여러 게이트를 만들 수 있다.

## 연습 2-6

입력이 세 개인 AND 게이트를 시뮬레이션하는 모델을 만들어 보자. 이 게이트는 세 입력이 모두 1일 때만 1을 출력한다.

In [ ]:
# 입력이 세 개인 AND 게이트: 000 ~ 111 여덟 가지 조합
X3 = torch.tensor([[float(b) for b in f'{i:03b}'] for i in range(8)])
Y3 = (X3.sum(dim=1, keepdim=True) == 3).float()      # 셋 다 1일 때만 1

model = train_gate(X3, Y3)
pred = (model(X3) > 0.5).float()
for x, p, y in zip(X3.tolist(), pred.flatten().tolist(), Y3.flatten().tolist()):
    print(f'입력 {[int(v) for v in x]} -> 예측 {int(p)} (정답 {int(y)})')
print(f'\n전체 정답 여부: {torch.equal(pred, Y3)}')

입력이 셋으로 늘어도 선형 계층의 입력 크기만 3으로 바꾸면 된다. 3차원 공간에서 (1,1,1) 한 점만 나머지 일곱 점과 평면 하나로 분리할 수 있으므로 여전히 퍼셉트론 하나로 해결된다.